# 03. Qwen3 Scorer QLoRA 훈련

고정 epoch 정책을 먼저 만들고 fold×seed registry를 plan-only로 검토한 다음,
명시적 스위치로 순차 훈련한다. 중단 산출물은 삭제하지 않는다.


## 1. 독립 Colab 부트스트랩


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 2. 훈련 파라미터와 fail-fast 검사


In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

EXPERIMENT_ID = f"qwen3_scorer_{RUN_NAMESPACE}"
# ALLOW_MODEL_DOWNLOAD도 registry signature의 일부다. plan 이후 resume까지 바꾸지 않는다.
SEEDS = [42]
SELECTED_FOLDS = [0, 1, 2, 3, 4]
# 이 notebook의 OOF 경로는 완전한 5-fold를 요구한다. fold 집합은 registry 생성 뒤
# immutable이므로 plan/execute 사이에 바꾸지 않는다.
if SELECTED_FOLDS != [0, 1, 2, 3, 4]:
    raise ValueError("정식 OOF 훈련은 SELECTED_FOLDS를 0~4 전체로 유지해야 합니다.")
FIXED_EPOCH = 3

FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
POLICY_PATH = PROJECT_ROOT / "artifacts" / "policies" / f"{EXPERIMENT_ID}_epoch{FIXED_EPOCH}.json"
REGISTRY_PATH = PROJECT_ROOT / "artifacts" / "models" / EXPERIMENT_ID / "registry.json"

CREATE_EPOCH_POLICY = False
PLAN_TRAINING = False
EXECUTE_TRAINING = False
RETRY_PARTIAL = False
VALIDATE_COMPLETE_REGISTRY = False
BUILD_SEED_OOF = False
BUILD_SEED_ENSEMBLE_OOF = False

require_paths(FOLDS_PATH)
require_model_revision(MODEL_REVISION, enabled=PLAN_TRAINING or EXECUTE_TRAINING)
require_bf16_gpu(enabled=EXECUTE_TRAINING)
print({"experiment": EXPERIMENT_ID, "seeds": SEEDS, "folds": SELECTED_FOLDS})


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


## 3. 고정 epoch 정책


In [ ]:
run_cli(
    "fixed epoch policy",
    CREATE_EPOCH_POLICY,
    "scripts/select_fixed_epoch.py",
    "--preselected-epoch", FIXED_EPOCH,
    "--reason", "outer fold 학습 전에 고정한 Colab epoch 정책",
    "--output", POLICY_PATH,
)
show_json(POLICY_PATH)


## 4. fold×seed plan-only registry


In [ ]:
def orchestration_arguments(*, execute: bool) -> list[object]:
    arguments: list[object] = [
        "scripts/orchestrate_scorer_training.py",
        "--experiment-id", EXPERIMENT_ID,
        "--config", "configs/scorer_qlora.yaml",
        "--data-config", "configs/data.yaml",
        "--folds-file", FOLDS_PATH,
        "--epoch-policy", POLICY_PATH,
        "--model-revision", MODEL_REVISION,
    ]
    for seed in SEEDS:
        arguments.extend(["--seed", seed])
    for fold in SELECTED_FOLDS:
        arguments.extend(["--fold", fold])
    if execute:
        arguments.append("--execute")
    if execute and RETRY_PARTIAL:
        arguments.append("--retry-partial")
    arguments.extend(download_flag)
    return arguments

if PLAN_TRAINING:
    require_paths(POLICY_PATH)
run_cli("training plan", PLAN_TRAINING, *orchestration_arguments(execute=False))
show_json(REGISTRY_PATH)


## 5. 실제 훈련

위 registry의 모든 command, fold, seed, fixed epoch, 출력 경로를 확인한 뒤에만
`EXECUTE_TRAINING=True`로 바꾼다. 중단 뒤에도 fold/seed 인자를 바꾸지 않고 동일한
full-registry command를 다시 실행한다. 검증된 완료 task는 건너뛰고 남은 task부터 이어간다.


In [ ]:
if EXECUTE_TRAINING:
    require_paths(POLICY_PATH, REGISTRY_PATH)
run_cli("execute fold training", EXECUTE_TRAINING, *orchestration_arguments(execute=True))


## 6. registry 완결성 검사


In [ ]:
run_cli(
    "validate complete registry",
    VALIDATE_COMPLETE_REGISTRY,
    "scripts/validate_run_registry.py",
    "--registry", REGISTRY_PATH,
    "--require-complete",
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{EXPERIMENT_ID}_registry.json",
)


## 7. fixed-epoch OOF 조립


In [ ]:
TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"

def checkpoint_dir(seed: int, fold: int) -> Path:
    return (
        PROJECT_ROOT / "artifacts" / "models" / EXPERIMENT_ID
        / f"seed_{seed}__fold_{fold}" / f"epoch_{FIXED_EPOCH}"
    )

seed_oof_paths = {}
for seed in SEEDS:
    output = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_seed{seed}_oof.jsonl"
    seed_oof_paths[seed] = output
    arguments: list[object] = [
        "scripts/build_oof.py", "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
    ]
    for fold in range(5):
        arguments.extend(["--pred", checkpoint_dir(seed, fold) / "oof.jsonl"])
    arguments.extend(["--output", output, "--model", f"{EXPERIMENT_ID}_seed{seed}"])
    run_cli(f"build seed {seed} OOF", BUILD_SEED_OOF, *arguments)

ensemble_oof = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_seed_ensemble_oof.jsonl"
ensemble_arguments: list[object] = [
    "scripts/build_seed_ensemble_oof.py", "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
]
for seed, path in seed_oof_paths.items():
    ensemble_arguments.extend(["--source", f"{seed}={path}"])
ensemble_arguments.extend(["--output", ensemble_oof, "--model", f"{EXPERIMENT_ID}_seed_ensemble"])
if BUILD_SEED_ENSEMBLE_OOF and len(SEEDS) < 2:
    raise ValueError("seed ensemble에는 서로 다른 seed OOF가 최소 2개 필요합니다.")
run_cli("build seed ensemble OOF", BUILD_SEED_ENSEMBLE_OOF, *ensemble_arguments)
print("next OOF:", ensemble_oof if len(SEEDS) > 1 else seed_oof_paths[SEEDS[0]])


## 8. 다음 단계

`04_calibration_ensemble_inference.ipynb`에서 raw OOF provenance를 다시 검증하고
calibration·precision·stacking 후보를 평가한다. outer-held 성능으로 고른 best epoch를
OOF에 사용하지 않는다.
